# Data Preparation

## 1. Import Libraries

In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split

## 2. Fetch Stock Data
Download AAPL and TSLA data from 2019-01-01 to 2024-01-01.

In [ ]:
stocks = ['AAPL', 'TSLA']
start_date = '2019-01-01'
end_date = '2024-01-01'

data = yf.download(stocks, start=start_date, end=end_date)
print(data.head())

## 3. Save Raw Data

In [ ]:
output_path = '../data/raw_stock_data.csv'
os.makedirs('../data', exist_ok=True)
data.to_csv(output_path)
print(f"Data saved to {output_path}")

## 4. Preprocess Data
Standardize and Clean.

In [ ]:
# Reload to handle potential multi-index issues or ensure consistency
try:
    df_raw = pd.read_csv('../data/raw_stock_data.csv', header=[0, 1], index_col=0)
    # Select AAPL for this example
    df = df_raw.xs('AAPL', axis=1, level=1).dropna()
except:
    # Fallback for single header
    df = pd.read_csv('../data/raw_stock_data.csv', index_col=0).dropna()

scaler = MinMaxScaler()
feature_cols = ['Open', 'High', 'Low', 'Close', 'Volume']
df_scaled = scaler.fit_transform(df[feature_cols])

## 5. Create Sequences and Labels

In [ ]:
def create_sequences(data, labels, window_size=60):
    X, y = [], []
    for i in range(len(data) - window_size):
        X.append(data[i:i+window_size])
        y.append(labels[i+window_size])
    return np.array(X), np.array(y)

# Binary labels: 1 if Close[t] > Close[t-1]
close_prices = df['Close'].values
labels = (np.roll(close_prices, -1) > close_prices).astype(int)
labels[-1] = 0 # Invalid last label

X, y = create_sequences(df_scaled, labels, window_size=60)
# Remove last invalid sample
X, y = X[:-1], y[:-1]

print(f"X shape: {X.shape}, y shape: {y.shape}")

## 6. Split and Save

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=False)
np.save('../data/X_train.npy', X_train)
np.save('../data/X_test.npy', X_test)
np.save('../data/y_train.npy', y_train)
np.save('../data/y_test.npy', y_test)
print("Preprocessing Done.")

## 7. Validate Data
Check class balance and feature distributions.

In [ ]:
# Check Class Balance
unique, counts = np.unique(y_train, return_counts=True)
balance = dict(zip(unique, counts))
print(f"Class Balance (Train): {balance}")

plt.figure(figsize=(6, 4))
plt.bar(balance.keys(), balance.values(), color=['red', 'green'])
plt.xticks([0, 1], ['Down', 'Up'])
plt.title('Class Balance in Training Set')
plt.show()

# Plot Feature Distributions
plt.figure(figsize=(10, 6))
plt.hist(X_train.flatten(), bins=50, color='blue', alpha=0.7)
plt.title('Feature Distribution (Scaled)')
plt.show()